**Cell 1**

<a href="https://colab.research.google.com/github/Karthi6559/Intent-Analysis/blob/Request001/MechInterp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [81]:
# Cell 2

!pip install -q transformer_lens transformers einops

zsh:1: command not found: pip


**Cell 3**

### 1. Environment Setup
We begin by installing the necessary libraries: `transformer_lens` for mechanistic interpretability and `transformers` for the underlying model weights.

In [82]:
# Cell 4

import numpy as np # used for mathematical operations on Array
import torch #
import transformers
import transformer_lens

# Basic environment check to ensure all libraries are correctly linked
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("transformer_lens imported successfully")

# Specifically import the HookedTransformer class, which is our main interface for interpretability
from transformer_lens import HookedTransformer
print("HookedTransformer import: OK")

numpy: 2.4.6
torch: 2.7.1
transformers: 5.12.1
transformer_lens imported successfully
HookedTransformer import: OK


**Cell 5**

### 2. Hook Function Definition
This cell defines our custom hook. A hook in `transformer_lens` is a function that takes the current `activation` (the tensor data) and a `hook` object (metadata). It allows us to view or modify data mid-computation.

In [83]:
# Cell 6

# Need to test this
# This function intercepts the tensor (activation) as it passes through a layer
def my_analysis_hook(activation, hook):
    # 'activation' is the actual data (tensor) flowing through the network at this moment
    # 'hook.name' tells you exactly which layer and part of the network we are in

    print(f"--- Triggered at: {hook.name} ---")
    print(f"Data shape: {activation.shape}")

    # Example analysis: Let's see the average magnitude of the data flowing through
    # .item() extracts the single value from the PyTorch Tensor as a standard Python float
    avg_magnitude = torch.mean(torch.abs(activation)).item()
    print(f"Average activation magnitude: {avg_magnitude:.4f}\n")

    # You MUST return the activation.
    # (If you wanted to edit the prompt's flow, you would modify the tensor here before returning it!)
    return activation

In [84]:
# Cell 7

# This function intercepts the tensor (activation) as it passes through a layer
def my_analysis_hook(activation, hook):
    # 'activation' is the actual data (tensor) flowing through the network at this moment
    # 'hook.name' tells you exactly which layer and part of the network we are in

    print(f"--- Triggered at: {hook.name} ---")
    print(f"Data shape: {activation.shape}")

    # Example analysis: Let's see the average magnitude of the data flowing through
    avg_magnitude = torch.mean(torch.abs(activation)).item()
    print(f"Average activation magnitude: {avg_magnitude:.4f}\n")

    # You MUST return the activation.
    # (If you wanted to edit the prompt's flow, you would modify the tensor here before returning it!)
    return activation

**Cell 8**

### 3. Model Initialization
Here we load the `gpt2-small` model and inspect its configuration, such as the number of layers and the size of the hidden dimension (`d_model`).

In [85]:
# Cell 9

# Check for GPU availability to speed up model processing
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Initialize an empty list to store multiple hook configurations if needed
fwd_hooks = []

# Load the pre-trained GPT-2 small model using HookedTransformer for easy internal access
model = HookedTransformer.from_pretrained("gpt2-small", device=device)

# Extract and print architectural parameters
num_layers = model.cfg.n_layers
print("Loaded GPT-2 small")
print("layers :", num_layers)
print("heads  :", model.cfg.n_heads)
print("d_model:", model.cfg.d_model)

device: cpu


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 21031.92it/s]


Loaded pretrained model gpt2-small into HookedTransformer
Loaded GPT-2 small
layers : 12
heads  : 12
d_model: 768


In [86]:
# Cell 10

# Loop through all layers to create a 'global' monitoring setup
for layer in range(num_layers):
    # Construct the hook point name (hook_resid_pre is the input to a transformer block)
    hook_name = f"blocks.{layer}.hook_resid_pre"

    # Register our analysis function to be called at each of these hook points
    fwd_hooks.append((hook_name, my_analysis_hook))

**Cell 11**

### 4. Running with Hooks
We define our prompt and run it through the model. Using `run_with_hooks` allows us to apply our analysis function to every layer's residual stream as the data passes through.

In [ ]:
# Cell 12

prompt = 'resource "aws_s3_bucket_acl" "example" {\n  bucket = aws_s3_bucket.example.id\n  acl    = "'

fwd_hooks = []
for layer in range(model.cfg.n_layers):
    hook_name = f"blocks.{layer}.hook_resid_pre"
    fwd_hooks.append((hook_name, my_analysis_hook))

logits = model.run_with_hooks(prompt, fwd_hooks=fwd_hooks)

print("logits shape:", logits.shape)

--- Triggered at: blocks.0.hook_resid_pre ---
Data shape: torch.Size([1, 42, 768])
Average activation magnitude: 0.1174

--- Triggered at: blocks.1.hook_resid_pre ---
Data shape: torch.Size([1, 42, 768])
Average activation magnitude: 1.0107

--- Triggered at: blocks.2.hook_resid_pre ---
Data shape: torch.Size([1, 42, 768])
Average activation magnitude: 1.1799

--- Triggered at: blocks.3.hook_resid_pre ---
Data shape: torch.Size([1, 42, 768])
Average activation magnitude: 1.4792

--- Triggered at: blocks.4.hook_resid_pre ---
Data shape: torch.Size([1, 42, 768])
Average activation magnitude: 1.6986

--- Triggered at: blocks.5.hook_resid_pre ---
Data shape: torch.Size([1, 42, 768])
Average activation magnitude: 1.8560

--- Triggered at: blocks.6.hook_resid_pre ---
Data shape: torch.Size([1, 42, 768])
Average activation magnitude: 2.0779

--- Triggered at: blocks.7.hook_resid_pre ---
Data shape: torch.Size([1, 42, 768])
Average activation magnitude: 2.3104

--- Triggered at: blocks.8.hook_

In [88]:
# Cell 13

#By default, TransformerLens's to_tokens method will automatically prepend a
#special BOS (Beginning of Sequence) token to the start of your prompt

tokens = model.to_tokens(prompt)

#In the context of your repository and the TransformerLens library,
#the line tokens = model.to_tokens(prompt) is used to convert your human-readable text
#string into a format that the neural network can understand.

# pattern_l0 = cache["pattern", 0]
# mlp_out_l0 = cache["mlp_out", 0]
# resid_post_l0 = cache["resid_post", 0]

print("tokens shape        :", tokens.shape)
# print("pattern_l0 shape    :", pattern_l0.shape) # Attention pattern
# print("mlp_out_l0 shape    :", mlp_out_l0.shape) # a output of the multi-layer perceptron
# print("resid_post_l0 shape :", resid_post_l0.shape) # Residual stream output

tokens shape        : torch.Size([1, 42])


In [89]:
# Cell 14

def final_logits(model, text):
    logits = model(text)
    return logits[0, -1]

def logit_diff(model, text, token_a, token_b):
    fl = final_logits(model, text)
    id_a = model.to_single_token(token_a)
    id_b = model.to_single_token(token_b)
    return (fl[id_a] - fl[id_b]).item()

prompt = 'resource "aws_s3_bucket_acl" "example" {\n  bucket = aws_s3_bucket.example.id\n  acl    = "'
# Reusing the globally initialized 'model'
score = logit_diff(model, prompt, " private" , " public")
print(f"Logit difference (private vs public): {score:.4f}")

Logit difference (private vs public): -2.2302


In [90]:
# Cell 15

SECURE_FULL   = 'resource "aws_s3_bucket_acl" "example" {\n  bucket = aws_s3_bucket.example.id\n  acl    = "private'
INSECURE_FULL = 'resource "aws_s3_bucket_acl" "example" {\n  bucket = aws_s3_bucket.example.id\n  acl    = "public'

_, cache_sec = model.run_with_cache(SECURE_FULL)
_, cache_ins = model.run_with_cache(INSECURE_FULL)

# Verify the last token of each is what you expect
print("Last token of SECURE_FULL  :", model.to_str_tokens(SECURE_FULL)[-1])
print("Last token of INSECURE_FULL:", model.to_str_tokens(INSECURE_FULL)[-1])

Last token of SECURE_FULL  : private
Last token of INSECURE_FULL: public


In [91]:
# Cell 16

import matplotlib.pyplot as plt

ALPHA = 20.0
probe_tokens = model.to_tokens(prompt)
secure_tok   = model.to_single_token(" private")
insecure_tok = model.to_single_token(" public")

baseline_ld = logit_diff(model, prompt, " private", " public")

steered_lds = []

for layer in range(model.cfg.n_layers):
    sv = (cache_sec[f"blocks.{layer}.hook_resid_pre"][0, -1, :]
        - cache_ins[f"blocks.{layer}.hook_resid_pre"][0, -1, :])

    def hook_fn(resid, hook, _sv=sv):
        resid[:, -1, :] = resid[:, -1, :] + ALPHA * _sv
        return resid

    steered_logits = model.run_with_hooks(
        probe_tokens,
        fwd_hooks=[(f"blocks.{layer}.hook_resid_pre", hook_fn)]
    )[0, -1, :]

    ld = (steered_logits[secure_tok] - steered_logits[insecure_tok]).item()
    steered_lds.append(ld)
    print(f"Layer {layer:2d}: {ld:+.4f}")

peak_layer = steered_lds.index(max(steered_lds))

plt.figure(figsize=(10, 5))
plt.plot(range(model.cfg.n_layers), steered_lds, "o-", color="#2196F3", label="Steered logit diff")
plt.axhline(baseline_ld, linestyle=":", color="#2196F3", label=f"Baseline: {baseline_ld:.4f}")
plt.axvline(peak_layer, linestyle="--", color="red", label=f"Peak layer: {peak_layer}")
plt.title("CWE-732 Activation Steering Sweep\n\" private\" vs \" public\" | GPT-2 Small")
plt.xlabel("Layer")
plt.ylabel("Logit Diff (private − public)")
plt.xticks(range(model.cfg.n_layers))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nPeak layer: {peak_layer}")
print(f"Baseline logit diff: {baseline_ld:.4f}")
print(f"Peak steered logit diff: {max(steered_lds):.4f}")

ModuleNotFoundError: No module named 'matplotlib'

**Cell 17**

### Targeted Activation Patching at the Identified Peak Layer
The sweep above identified `peak_layer` as the residual-stream point where the contrastive (secure − insecure) vector most increases the model's preference for `" private"` over `" public"`. Here we apply a single, targeted patch at exactly that point — not a sweep — and verify the effect is:

1. **Measurable**: clean vs. patched logit diff and `P(" private")` are compared directly.
2. **Reproducible**: the patched forward pass is re-run multiple times on the same input (must be bit-identical), and re-tested on held-out prompts (different bucket names) the steering vector was never built from, to confirm the shift generalises rather than overfitting to one prompt.

In [ ]:
# Cell 18

patch_hook_point = f"blocks.{peak_layer}.hook_resid_pre"
patch_sv = (cache_sec[patch_hook_point][0, -1, :] - cache_ins[patch_hook_point][0, -1, :])

def patch_resid(resid, hook, _sv=patch_sv, _alpha=ALPHA):
    resid[:, -1, :] = resid[:, -1, :] + _alpha * _sv
    return resid

# Held-out prompts: same ACL pattern, different bucket names. The steering vector
# was derived only from the "example" bucket, so any shift here is genuine
# generalisation rather than an artifact of the exact prompt it was built from.
held_out_prompts = [
    'resource "aws_s3_bucket_acl" "example" {\n  bucket = aws_s3_bucket.example.id\n  acl    = "',
    'resource "aws_s3_bucket_acl" "data_lake" {\n  bucket = aws_s3_bucket.data_lake.id\n  acl    = "',
    'resource "aws_s3_bucket_acl" "logs" {\n  bucket = aws_s3_bucket.logs.id\n  acl    = "',
]

print(f"Patching at: {patch_hook_point}  (ALPHA={ALPHA})\n")
print(f"{'Bucket':<12} {'Clean LD':>10} {'Patched LD':>11} {'Delta':>8} {'P(priv) clean':>14} {'P(priv) patched':>16}")

deltas = []
for p in held_out_prompts:
    toks = model.to_tokens(p)
    bucket_name = p.split('"')[3]

    clean_logits = model(toks)[0, -1, :]
    clean_ld = (clean_logits[secure_tok] - clean_logits[insecure_tok]).item()
    clean_p_secure = torch.softmax(clean_logits, dim=-1)[secure_tok].item()

    patched_logits = model.run_with_hooks(toks, fwd_hooks=[(patch_hook_point, patch_resid)])[0, -1, :]
    patched_ld = (patched_logits[secure_tok] - patched_logits[insecure_tok]).item()
    patched_p_secure = torch.softmax(patched_logits, dim=-1)[secure_tok].item()

    deltas.append(patched_ld - clean_ld)
    print(f"{bucket_name:<12} {clean_ld:>10.4f} {patched_ld:>11.4f} {patched_ld - clean_ld:>+8.4f} "
          f"{clean_p_secure:>14.4f} {patched_p_secure:>16.4f}")

# Reproducibility check 1: re-run the patched forward pass on the same input several
# times. Model and hook are both deterministic, so the logit diff must be identical.
repeat_lds = []
for _ in range(3):
    toks = model.to_tokens(held_out_prompts[0])
    l = model.run_with_hooks(toks, fwd_hooks=[(patch_hook_point, patch_resid)])[0, -1, :]
    repeat_lds.append((l[secure_tok] - l[insecure_tok]).item())

assert max(repeat_lds) - min(repeat_lds) < 1e-6, "Patch is not deterministic across repeated runs!"

print(f"\nRepeated patched logit diff on same input (determinism check): {repeat_lds}")
print(f"Average shift toward secure token across {len(held_out_prompts)} held-out prompts: {sum(deltas)/len(deltas):+.4f}")
print(f"All {len(deltas)} held-out prompts shifted toward secure: {all(d > 0 for d in deltas)}")

In [ ]:
# Cell 19

import matplotlib.pyplot as plt

# Re-cache the clean probe (no steering) to read intermediate layers
_, probe_cache = model.run_with_cache(probe_tokens)

secure_probs   = []
insecure_probs = []

for layer in range(model.cfg.n_layers):
    # Residual stream after this layer, last token position
    # Shape kept as [1, 1, d_model] so ln_final and unembed work correctly
    resid = probe_cache[f"blocks.{layer}.hook_resid_post"][0:1, -1:, :]

    # CRITICAL: apply the final layer norm before unembedding
    normed = model.ln_final(resid)            # [1, 1, d_model]
    logits = model.unembed(normed).squeeze()  # [d_vocab]
    probs  = torch.softmax(logits, dim=-1)

    sp = probs[secure_tok].item()
    ip = probs[insecure_tok].item()
    secure_probs.append(sp)
    insecure_probs.append(ip)
    lead = "SECURE leads" if sp > ip else "insecure leads"
    print(f"Layer {layer:2d}  P(private)={sp:.4f}  P(public)={ip:.4f}  {lead}")

# Where does secure first overtake insecure (if ever)?
crossover = next((i for i in range(model.cfg.n_layers)
                  if secure_probs[i] > insecure_probs[i]), None)

plt.figure(figsize=(10, 5))
plt.plot(range(model.cfg.n_layers), secure_probs, "o-", color="#4CAF50", label='P(" private") — secure')
plt.plot(range(model.cfg.n_layers), insecure_probs, "s-", color="#F44336", label='P(" public") — insecure')
if crossover is not None:
    plt.axvline(crossover, linestyle="--", color="#4CAF50", label=f"Crossover: layer {crossover}")
else:
    plt.text(model.cfg.n_layers*0.4, max(max(secure_probs), max(insecure_probs))*0.8,
             "No crossover — insecure preferred throughout",
             color="darkred", ha="center",
             bbox=dict(boxstyle="round", facecolor="mistyrose"))
plt.title("CWE-732 Logit Lens\nWhen does each token become readable? | GPT-2 Small")
plt.xlabel("Layer")
plt.ylabel("Token probability")
plt.xticks(range(model.cfg.n_layers))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nCrossover layer: {crossover}")
if crossover is None:
    print("P(private) never overtakes P(public) — the insecure preference is")
    print("present in the residual stream throughout, corroborating the -1.32 baseline.")

In [ ]:
# Cell 20

import matplotlib.pyplot as plt
import torch

PEAK_LAYER = 1          # from your steering sweep
ALPHA      = 20.0       # same scale as the sweep

prompt_tokens = model.to_tokens(prompt)
secure_tok    = model.to_single_token(" private")
insecure_tok  = model.to_single_token(" public")

# Raw contrastive steering vector at the peak layer (do NOT normalise)
sv = (cache_sec[f"blocks.{PEAK_LAYER}.hook_resid_pre"][0, -1, :]
    - cache_ins[f"blocks.{PEAK_LAYER}.hook_resid_pre"][0, -1, :])

def steer_hook(resid, hook, _sv=sv):
    resid[:, -1, :] = resid[:, -1, :] + ALPHA * _sv
    return resid

pattern_hook = f"blocks.{PEAK_LAYER}.attn.hook_pattern"

# Clean run
_, clean_cache = model.run_with_cache(prompt_tokens)
clean_attn = clean_cache[pattern_hook][0]          # [n_heads, q_pos, k_pos]

# Steered run — hook at hook_resid_pre (CRITICAL), capture same patterns
with model.hooks(fwd_hooks=[(f"blocks.{PEAK_LAYER}.hook_resid_pre", steer_hook)]):
    _, steered_cache = model.run_with_cache(prompt_tokens)
steered_attn = steered_cache[pattern_hook][0]

# Per-head L1 difference across all query/key positions
head_diffs = (steered_attn - clean_attn).abs().mean(dim=(-1, -2))   # [n_heads]
ranked = torch.argsort(head_diffs, descending=True)

print(f"Attention pattern shift at layer {PEAK_LAYER} (clean vs steered):")
for rank, h in enumerate(ranked):
    h = h.item()
    tag = "  <- ACTIVE" if rank < 3 else ""
    print(f"  Head {h:2d}: dL1 = {head_diffs[h].item():.5f}{tag}")

active_heads = [ranked[i].item() for i in range(3)]
print(f"\nTop 3 active heads at layer {PEAK_LAYER}: {active_heads}")

In [ ]:
# Cell 21

n_heads = model.cfg.n_heads
colours = ["#E91E63" if i in active_heads else "#90A4AE" for i in range(n_heads)]

plt.figure(figsize=(10, 5))
plt.bar(range(n_heads), head_diffs.cpu().numpy(), color=colours, edgecolor="white")
for h in active_heads:
    plt.text(h, head_diffs[h].item(), f"H{h}", ha="center", va="bottom",
             color="#E91E63", fontweight="bold")
plt.title(f"CWE-732 Attention Analysis — Head Shifts at Layer {PEAK_LAYER}\n(Clean vs Steered) | GPT-2 Small")
plt.xlabel("Attention head")
plt.ylabel("Mean |delta attention|")
plt.xticks(range(n_heads))
plt.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 22

token_labels = model.to_str_tokens(prompt)
W = min(12, len(token_labels))
labels = token_labels[-W:]

fig, axes = plt.subplots(1, len(active_heads), figsize=(6*len(active_heads), 5))
if len(active_heads) == 1:
    axes = [axes]
fig.suptitle(f"Attention Pattern Differences at Layer {PEAK_LAYER} (Steered - Clean)\n"
             "Red = attends MORE after secure steering, blue = less", fontsize=12)

for ax, head in zip(axes, active_heads):
    diff = (steered_attn[head] - clean_attn[head]).cpu().numpy()[-W:, -W:]
    im = ax.imshow(diff, cmap="RdBu_r", vmin=-0.3, vmax=0.3, aspect="auto")
    ax.set_title(f"Head {head}  (dL1={head_diffs[head].item():.4f})")
    ax.set_xticks(range(W)); ax.set_yticks(range(W))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel("Key position (attended to)")
    ax.set_ylabel("Query position (attending)")
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 23

clean_code = "for i in range(5):\n    print(i)"
corrupted_code = "for i in range(5):\nprint(i)"

if 'model' in globals():
    print("Clean code logit diff    :", logit_diff(model, clean_code, " print", " return"))
    print("Corrupted code logit diff:", logit_diff(model, corrupted_code, " print", " return"))
else:
    print("Model not found. Please run the cell above first.")

In [ ]:
# Cell 24

from transformer_lens.model_bridge import TransformerBridge

# Instead of booting a new model, we can interface with our existing 'model'
# bridge = TransformerBridge.boot_transformers("gpt2", device="cpu")

# Using the model we already have in memory
logits, cache = model.run_with_cache("Hello World")
print("Run complete using existing model.")

**Cell 25**

### Target Hooking Practice
This example demonstrates how to attach a hook to a specific component (the residual stream) of a single layer (Layer 5) for one specific run.

In [ ]:
# Cell 26

def single_layer_hook(activation, hook):
    print(f"Captured activations at: {hook.name}")
    print(f"Shape: {activation.shape} (batch, position, d_model)")
    print(f"Mean activation: {activation.mean().item():.4f}")
    return activation

target_layer = 5
target_hook_name = f"blocks.{target_layer}.hook_resid_post"

print(f"--- Running model with hook on layer {target_layer} ---")
# Reusing global model
logits = model.run_with_hooks(
    clean_code,
    fwd_hooks=[(target_hook_name, single_layer_hook)]
)
print("Run complete.")

**Cell 27**

### Detailed Single-Layer Hook Analysis
This version captures the internal activations into a dictionary and breaks down the activation mean for every individual token in your prompt.

In [ ]:
# Cell 28

hook_data = {}

def detailed_inspect_hook(activation, hook):
    hook_data['activation'] = activation.detach().clone()
    hook_data['name'] = hook.name
    return activation

# Inspect Layer 5 specifically for this code prompt
model.run_with_hooks(
    prompt,
    fwd_hooks=[("blocks.5.hook_resid_pre", detailed_inspect_hook)]
)

act = hook_data['activation'][0]
tokens = model.to_str_tokens(prompt)
token_means = act.mean(dim=-1)

print(f"Analysis of Python Code Prompt (Layer 5): {hook_data['name']}")
print(f"{'Pos':>4} | {'Token':>15} | {'Mean Act':>10}")
print('-' * 35)
for pos, (tok, val) in enumerate(zip(tokens, token_means)):
    print(f'{pos:>4} | {repr(tok):>15} | {val.item():>10.4f}')

In [ ]:
# Cell 29

print("--- Deep Dive: Layer 5 Contextual Representations ---")
print("At Layer 5 (the middle of GPT-2), the model has moved past just recognizing")
print("characters and is now building contextual representations. For example,")
print("it 'knows' that the token 'i' inside a for loop is an iterator, not just the letter 'i'.")
print("------------------------------------------------------")

In [ ]:
# Cell 30

import plotly.express as px

# 'act' was captured in the previous cell and has shape [sequence_length, 768]
# We'll visualize the first 100 dimensions for clarity
fig = px.imshow(
    act[:, :100].cpu().numpy(),
    labels=dict(x="Model Dimension (Neurons)", y="Token", color="Activation"),
    x=list(range(100)),
    y=tokens,
    title="Heatmap of Internal Activations (Layer 5, First 100 Dimensions)",
    color_continuous_scale="RdBu_r",
    aspect="auto"
)
fig.show()

In [ ]:
# Cell 31

from transformer_lens import HookedTransformer
import torch

# Check if the model is already in memory; if not, load it from the Hugging Face hub
if 'model' not in globals():
    model = HookedTransformer.from_pretrained('gpt2-small')

# Create a dictionary to 'leak' data out of the temporary hook function
hook_data = {}

# This function acts as a 'tap' on the neural network's internal wires
def inspect_hook(activation, hook):
    # We clone the activation tensor so it doesn't get overwritten during the rest of the forward pass
    hook_data['activation'] = activation.detach().clone()
    hook_data['name'] = hook.name
    # Crucial: Hooks must return the original (or modified) activation to let the model continue
    return activation

# Define the prompt we want to analyze
prompt = "Developer request: write a login function that securely stores passwords.\nCode:\n"

# Execute the forward pass, attaching our tap to the residual stream of Layer 5
model.run_with_hooks(
    prompt,
    fwd_hooks=[('blocks.5.hook_resid_pre', inspect_hook)]
)

# Process the captured data
# hook_data['activation'] has shape [batch, sequence_length, model_dimension]
# We select index 0 to ignore the batch dimension
act = hook_data['activation'][0]

# Convert the raw text into the specific token strings used by the model
tokens = model.to_str_tokens(prompt)

# Calculate the average activation 'energy' per token across all 768 model dimensions
token_means = act.mean(dim=-1)

# Print the results in a readable table
print(f"Captured at: {hook_data['name']}\n")
print(f"{'Pos':>4} | {'Token':>15} | {'Mean Act':>10}")
print('-' * 35)
for pos, (tok, val) in enumerate(zip(tokens, token_means)):
    print(f'{pos:>4} | {repr(tok):>15} | {val.item():>10.4f}')

**Cell 32**

### 6. Custom Code Generation
This cell sets the prompt to "Generate a code " and uses the model to generate a following code block.

In [ ]:
# Cell 33

developer_prompt = "Generate a code "

# Reusing the existing 'model' variable for generation
generated_output = model.generate(
    developer_prompt,
    max_new_tokens=30,
    temperature=0.7,
    verbose=False
)

print("\n--- Resulting Output ---")
print(generated_output)

**Cell 34**

### 5. Summary Analysis
This final block captures activations from a specific layer (Layer 5) and calculates the average activation for every single token. This helps identify which words in the prompt are 'activating' that specific part of the transformer's brain.

**Cell 35**

**Cell 36**

### Revised Project Summary: Mechanistic Interpretability

We have expanded our analysis to include deeper comparative studies of how GPT-2 processes Python structures:

1.  **Global vs. Local Monitoring**: While our initial setup monitored every layer to see general activation trends, we successfully transitioned to **Targeted Hooking**. This allowed us to isolate Layer 5 to see exactly how it represents specific tokens like `for` and `print`.
2.  **Logit Difference Analysis**: We introduced a quantitative way to measure model preference. By comparing the 'logits' for `print` versus `return`, we observed how the model's confidence shifts based on the preceding code context.
3.  **Causal Sensitivity**: By testing 'Clean' vs 'Corrupted' code (fixing or removing indentation), we began to see how sensitive the model's internal states are to syntactical correctness.
4.  **Visualizing Internals**: Using heatmaps of the residual stream, we moved from looking at single average numbers to seeing the 'fingerprint' of activations across 768 dimensions for every word in our code snippet.